In [10]:
!pip install fastapi uvicorn pyngrok nest-asyncio python-multipart


# Import Libraries

In [12]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import numpy as np
import cv2
import nest_asyncio

from fastapi import FastAPI, UploadFile, File
from pyngrok import ngrok
import uvicorn
from threading import Thread

# Load Model

In [13]:
model = load_model("rice_classification_model.h5")

print("Model Loaded Successfully")

Model Loaded Successfully


# Class Names

In [14]:
classes = [
    "Arborio",
    "Basmati",
    "Ipsala",
    "Jasmine",
    "Karacadag"
]

# Create API

In [15]:
app = FastAPI()

@app.get("/")
def home():
    return {"message": "Rice Classification API Running"}

# Prediction API

In [16]:
@app.post("/predict")
async def predict(file: UploadFile = File(...)):

    contents = await file.read()

    with open("temp.jpg", "wb") as f:
        f.write(contents)

    img = cv2.imread("temp.jpg", cv2.IMREAD_GRAYSCALE)

    img = cv2.resize(img, (200, 200))
    img = img / 255.0

    img = img.reshape(1, 200, 200, 1)

    prediction = model.predict(img)

    class_index = np.argmax(prediction)

    rice_type = classes[class_index]

    confidence = float(np.max(prediction))

    return {
        "rice_type": rice_type,
        "confidence": confidence
    }

# Start API

In [17]:
nest_asyncio.apply()

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

Thread(target=run).start()

# Create Public URL

In [18]:
public_url = ngrok.connect(8000)

print(public_url)

NgrokTunnel: "https://unsubversive-monica-semistiff.ngrok-free.dev" -> "http://localhost:8000"
